# End-to-End Medallion Pipeline Orchestrator

Executes the full Lakehouse pipeline in order for the specified environment:
1. **`01_Config`**: Initializes catalogs (`*_dev` or `*_prod`), schemas, and volumes.
2. **`01_Extract_API_Task`**: Extracts previous day's REST API order records into the landing volume.
3. **`00_Data_Generator`**: Generates synthetic baseline data (or snapshot via shallow clone).
4. **`02_Bronze_Layer`**: Streams raw data (Parquet & JSON) via Auto Loader into Bronze tables.
5. **`03_Silver_Layer`**: Conforms domains, deduplicates API orders, enforces PII tags, and runs SCD Type 2 MERGE.
6. **`04_Gold_Layer`**: Builds Kimball Star Schema, Power BI date dimension, and daily business KPIs.


In [0]:
# Orchestrator Configuration Widgets
dbutils.widgets.dropdown("target_env", "dev", ["dev", "prod", "qa"], "Target Environment (dev | prod | qa)")
dbutils.widgets.dropdown("clone_from_prod", "false", ["false", "true"], "Clone Tables from Prod (Shallow Clone for Dev)")
dbutils.widgets.dropdown("generate_sample_data", "true", ["true", "false"], "Generate Sample Data")
dbutils.widgets.dropdown("batch_mode", "initial", ["initial", "update"], "Data Batch Mode (initial | update)")
dbutils.widgets.dropdown("extract_api_data", "true", ["true", "false"], "Extract REST API Data")
dbutils.widgets.text("api_batch_date", "yesterday", "API Batch Date (YYYY-MM-DD or yesterday)")

target_env = dbutils.widgets.get("target_env").lower().strip()
clone_prod = dbutils.widgets.get("clone_from_prod")
gen_data = dbutils.widgets.get("generate_sample_data")
batch_mode = dbutils.widgets.get("batch_mode")
extract_api = dbutils.widgets.get("extract_api_data")
api_date = dbutils.widgets.get("api_batch_date")

print(f"[ORCHESTRATOR] Starting pipeline execution in: {target_env.upper()}")


In [0]:
import time
from datetime import datetime

def run_step(notebook_name, args=None):
    step_args = {"target_env": target_env}
    if args:
        step_args.update(args)
    print(f"\n{'='*70}")
    print(f"[ORCHESTRATOR] Running: {notebook_name} ({target_env.upper()}) at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}")
    t0 = time.time()
    result = dbutils.notebook.run(notebook_name, 3600, step_args)
    elapsed = time.time() - t0
    print(f"✓ Completed: {notebook_name} in {elapsed:.2f} seconds.")
    return notebook_name, elapsed

execution_log = []

# Step 1: Environment & Config
execution_log.append(run_step("01_Config"))

# Step 1.5: Optional Shallow Clone from Prod into Dev
if target_env == "dev" and clone_prod == "true":
    print("[ORCHESTRATOR] Triggering zero-copy shallow clone from Prod to Dev...")
    %run ./01_Config $target_env="dev"
    clone_prod_to_dev("prod", "dev")

# Step 2: REST API Extraction (Previous day JSON records)
if extract_api == "true":
    execution_log.append(run_step("01_Extract_API_Task", {"p_date": api_date}))

# Step 3: Data Generator (optional baseline)
if gen_data == "true":
    execution_log.append(run_step("00_Data_Generator", {"batch_mode": batch_mode}))

# Step 4: Bronze Auto Loader Ingestion (Parquet & JSON)
execution_log.append(run_step("02_Bronze_Layer", {"p_dataset": "all"}))

# Step 5: Silver Conformed, Deduplicated & SCD Type 2
execution_log.append(run_step("03_Silver_Layer"))

# Step 6: Gold Star Schema, Daily KPIs & Business Marts
execution_log.append(run_step("04_Gold_Layer"))

print("\n" + "="*70)
print(f"[ORCHESTRATOR] Pipeline Execution Summary ({target_env.upper()}): ")
print("="*70)
for nb, duration in execution_log:
    print(f"  • {nb:<25} : {duration:.2f} s")
total_time = sum(d for _, d in execution_log)
print(f"\nTotal Pipeline Execution Time: {total_time:.2f} s ({total_time/60:.2f} min)")
